# Bronze Ingestion — London Weather

Ingest current London weather conditions from Open-Meteo.

This notebook:

1. Loads weather source configuration.
2. Calls the Open-Meteo API.
3. Validates the response.
4. Lands the original JSON response.
5. Appends the response to the Bronze Delta table.

**Source:** Open-Meteo  
**Target:** `workspace.urbanpulse_bronze.weather`

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import ingestion components

In [0]:
import uuid
from urllib.parse import urlencode

from urbanpulse.ingestion.api_clients import ApiClient
from urbanpulse.ingestion.bronze import write_raw_bronze
from urbanpulse.ingestion.landing import land_json
from urbanpulse.utils.config import load_yaml

## 3. Load weather configuration

Weather coordinates, timezone, and requested variables are maintained in `conf/sources.yml`.

In [0]:
CONFIG_PATH = (
    PROJECT_ROOT
    / "conf"
    / "sources.yml"
)

config = load_yaml(
    str(CONFIG_PATH)
)

weather_config = (
    config["open_meteo"]["weather"]
)

BASE_URL = config["open_meteo"]["base_url"]
ENDPOINT = weather_config["endpoint"]

LATITUDE = weather_config["latitude"]
LONGITUDE = weather_config["longitude"]
TIMEZONE = weather_config["timezone"]

CURRENT_VARIABLES = (
    weather_config["current_variables"]
)

BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "weather"
)

LANDING_PATH = (
    "/Volumes/workspace/"
    "urbanpulse_meta/"
    "landing"
)

## 4. Build the Open-Meteo request

Query parameters define the location, timezone, and weather variables returned by the API.

In [0]:
request_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "timezone": TIMEZONE,
    "current": ",".join(
        CURRENT_VARIABLES
    ),
}

print(request_params)

## 5. Request current London weather

In [0]:
client = ApiClient(
    base_url=BASE_URL
)

payload, status_code = client.get(
    endpoint=ENDPOINT,
    params=request_params,
)

request_id = str(
    uuid.uuid4()
)

print(f"HTTP status: {status_code}")
print(f"Request ID: {request_id}")

## 6. Inspect the Open-Meteo response

The response contains location metadata, units, and a `current` object containing the requested weather measurements.

In [0]:
{
  "latitude": 51.5,
  "longitude": -0.12,
  "timezone": "Europe/London",
  "current_units": {
    "time": "iso8601",
    "temperature_2m": "°C"
  },
  "current": {
    "time": "2026-08-23T22:45",
    "temperature_2m": 18.2,
    "relative_humidity_2m": 72,
    "precipitation": 0.0
  }
}

## 7. Validate the source response

The pipeline requires location metadata and a populated `current` weather object before the response can enter Bronze.

In [0]:
if status_code != 200:
    raise RuntimeError(
        f"Open-Meteo returned HTTP "
        f"{status_code}"
    )

if not isinstance(payload, dict):
    raise TypeError(
        "Expected Open-Meteo response "
        "to be a dictionary"
    )

required_top_level_fields = {
    "latitude",
    "longitude",
    "timezone",
    "current",
}

missing_fields = (
    required_top_level_fields
    - set(payload.keys())
)

if missing_fields:
    raise ValueError(
        "Open-Meteo response is missing "
        f"fields: {sorted(missing_fields)}"
    )

current_weather = payload["current"]

if not isinstance(current_weather, dict):
    raise TypeError(
        "'current' must be a dictionary"
    )

if "time" not in current_weather:
    raise ValueError(
        "Current weather does not "
        "contain a timestamp"
    )

print(
    "Weather source validation passed."
)

In [0]:
missing_variables = [
    variable
    for variable in CURRENT_VARIABLES
    if variable not in current_weather
]

if missing_variables:
    raise ValueError(
        "Open-Meteo response is missing "
        f"variables: {missing_variables}"
    )

print(
    f"Validated "
    f"{len(CURRENT_VARIABLES)} "
    "weather variables."
)

## 9. Build traceable source metadata

Include the query parameters in `source_endpoint` so the Bronze record contains enough information to reproduce the API request.

In [0]:
source_endpoint = (
    f"{ENDPOINT}?"
    f"{urlencode(request_params)}"
)

print(source_endpoint)

## 10. Land the original JSON response

Retain the complete Open-Meteo response in the Unity Catalog landing Volume.

In [0]:
landing_file = land_json(
    payload=payload,
    base_path=LANDING_PATH,
    source="open_meteo",
    dataset="weather",
    request_id=request_id,
)

print(
    f"Raw file landed: "
    f"{landing_file}"
)

## 11. Append the weather snapshot to Bronze

Each execution represents one weather observation snapshot.

In [0]:
write_raw_bronze(
    spark=spark,
    payload=payload,
    request_id=request_id,
    source="open_meteo",
    dataset="weather",
    source_endpoint=source_endpoint,
    http_status=status_code,
    table_name=BRONZE_TABLE,
)

print(
    f"Bronze weather ingestion "
    f"completed: {request_id}"
)

## 12. Verify the raw landing file

In [0]:
landing_parent = str(
    Path(landing_file).parent
)

display(
    dbutils.fs.ls(
        landing_parent
    )
)

## 13. Verify the Bronze weather table

In [0]:
%sql
SELECT
    request_id,
    source,
    dataset,
    source_endpoint,
    ingested_at,
    http_status,
    LENGTH(payload) AS payload_size
FROM workspace.urbanpulse_bronze.weather
ORDER BY ingested_at DESC;

## 14. Inspect the stored weather payload

In [0]:
%sql
SELECT
    COUNT(*) AS weather_snapshots,
    MIN(ingested_at) AS first_ingestion,
    MAX(ingested_at) AS latest_ingestion
FROM workspace.urbanpulse_bronze.weather;